## 🎮 Data Abstraction Quest — Gamified Review

Test your knowledge in an interactive RPG! Explore the world below and talk to 5 NPC teachers — each one quizzes you on a different concept from this lesson.

| Key | Action |
|-----|--------|
| **WASD** | Move your character |
| **E** | Talk to an NPC (walk into them first) |

| NPC | Concept | Topic |
|-----|---------|-------|
| List Keeper (wizard) | Lists | Ordered sequences of elements |
| Index Oracle (robot) | Indexing | 0-based access, string indexing |
| Loop Sage (octocat) | Loops + Lists | Computing with iteration |
| Abstraction Architect (penguin) | Why Abstraction? | Managing complexity |
| AP Guardian (nerd) | AP Exam | 1-based indexing, pseudocode |

Complete all 5 to become **Master of Abstraction!** Watch your score in the top-right HUD.

In [2]:
%%js

// GAME_RUNNER: Explore the Data Kingdom! Walk to each NPC (WASD) and press E to answer quiz questions on Data Abstraction. Score 5/5 to become the Master of Abstraction!

// ─── Imports (GameEngine v1 — no engine files modified) ───
import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';

// ═══════════════════════════════════════════════════════════
//  SRP Module 1: QuizEngine — manages questions & grading
// ═══════════════════════════════════════════════════════════
var QuizEngine = {
  banks: {
    'List Keeper': [
      { q: "What is a list in programming?", choices: ["A single variable","An ordered sequence of elements","A function","A loop"], answer: 1 },
      { q: "In Python, how do you create a list?", choices: ["list = (1,2,3)","list = {1,2,3}","list = [1,2,3]","list = <1,2,3>"], answer: 2 },
      { q: "Lists can contain:", choices: ["Only numbers","Only strings","Only booleans","Different types of elements"], answer: 3 }
    ],
    'Index Oracle': [
      { q: "What index is the FIRST element in Python/JS?", choices: ["1","0","-1","None"], answer: 1 },
      { q: "Given word = 'hello', what is word[1]?", choices: ["'h'","'e'","'l'","'o'"], answer: 1 },
      { q: "On the AP exam, lists are:", choices: ["0-indexed","1-indexed","2-indexed","Not indexed"], answer: 1 }
    ],
    'Loop Sage': [
      { q: "scores=[85,92,78,90]; What is sum/len?", choices: ["345","85","86.25","90"], answer: 2 },
      { q: "Why use a loop with a list?", choices: ["It is slower but safer","It manages complexity and is reusable","Loops are required by Python","There is no advantage"], answer: 1 },
      { q: "Which keyword iterates over a JS array?", choices: ["for...of","for...at","each...in","loop...over"], answer: 0 }
    ],
    'Abstraction Architect': [
      { q: "Data abstraction means:", choices: ["Deleting unused variables","Treating collections as a single unit","Making code longer","Avoiding lists"], answer: 1 },
      { q: "Which is a benefit of data abstraction?", choices: ["Harder to read code","Easier maintenance","More memory usage","Slower programs"], answer: 1 },
      { q: "Abstraction lets you manipulate a group using:", choices: ["Individual memory addresses","A single name like scores","Only print statements","Manual indexing of every element"], answer: 1 }
    ],
    'AP Guardian': [
      { q: "On the AP CSP exam, what index is the first list element?", choices: ["0","1","2","-1"], answer: 1 },
      { q: "A string is:", choices: ["An unordered set","An ordered sequence of characters","A number type","A boolean"], answer: 1 },
      { q: "Data abstractions can be created using:", choices: ["Only dictionaries","Only integers","Lists","Only functions"], answer: 2 }
    ]
  },
  getQuestion: function(npcId) {
    var bank = this.banks[npcId];
    if (!bank || bank.length === 0) return null;
    return bank[Math.floor(Math.random() * bank.length)];
  },
  check: function(question, choiceIndex) {
    return choiceIndex === question.answer;
  }
};

// ═══════════════════════════════════════════════════════════
//  SRP Module 2: ScoreTracker — tracks which NPCs are beaten
// ═══════════════════════════════════════════════════════════
var ScoreTracker = {
  _done: {},
  total: 5,
  mark: function(npcId) { this._done[npcId] = true; },
  isComplete: function(npcId) { return this._done[npcId] === true; },
  score: function() { return Object.keys(this._done).length; },
  allDone: function() { return Object.keys(this._done).length >= this.total; },
  reset: function() { this._done = {}; }
};

// ═══════════════════════════════════════════════════════════
//  SRP Module 3: QuizOverlay — renders DOM quiz popup
// ═══════════════════════════════════════════════════════════
var QuizOverlay = {
  active: false,
  _escHandler: null,

  show: function(npcId, npcGreeting) {
    if (this.active) return;
    this.active = true;

    if (ScoreTracker.isComplete(npcId)) {
      this._showMessage(npcId, 'You already answered my question correctly! Move on, scholar!', '#4CAF50');
      return;
    }

    var question = QuizEngine.getQuestion(npcId);
    if (!question) { this.active = false; return; }
    var self = this;

    var overlay = document.createElement('div');
    overlay.id = 'quiz-overlay';
    overlay.style.cssText = 'position:fixed;top:0;left:0;width:100%;height:100%;background:rgba(0,0,0,0.85);display:flex;align-items:center;justify-content:center;z-index:10000;font-family:Segoe UI,sans-serif;';

    var card = document.createElement('div');
    card.style.cssText = 'background:linear-gradient(135deg,#1a1a2e,#16213e);color:#fff;border-radius:16px;padding:32px;max-width:520px;width:90%;box-shadow:0 0 40px rgba(100,100,255,0.3);border:2px solid #6c63ff;text-align:center;';

    var header = document.createElement('h2');
    header.textContent = npcId + ' asks:';
    header.style.cssText = 'color:#6c63ff;margin:0 0 8px 0;font-size:1.3em;';
    card.appendChild(header);

    var flavor = document.createElement('p');
    flavor.textContent = npcGreeting;
    flavor.style.cssText = 'color:#aaa;font-style:italic;margin:0 0 16px 0;font-size:0.9em;';
    card.appendChild(flavor);

    var qText = document.createElement('p');
    qText.style.cssText = 'font-size:1.1em;line-height:1.5;margin-bottom:20px;white-space:pre-wrap;';
    qText.textContent = question.q;
    card.appendChild(qText);

    var btnBox = document.createElement('div');
    btnBox.style.cssText = 'display:flex;flex-direction:column;gap:10px;';
    question.choices.forEach(function(choice, idx) {
      var btn = document.createElement('button');
      btn.textContent = choice;
      btn.style.cssText = 'padding:12px 16px;border:2px solid #6c63ff;border-radius:10px;background:transparent;color:#fff;font-size:1em;cursor:pointer;transition:all 0.2s ease;';
      btn.onmouseenter = function() { btn.style.background = '#6c63ff'; btn.style.transform = 'scale(1.03)'; };
      btn.onmouseleave = function() { btn.style.background = 'transparent'; btn.style.transform = 'scale(1)'; };
      btn.onclick = function() { self._handleAnswer(npcId, question, idx, overlay); };
      btnBox.appendChild(btn);
    });
    card.appendChild(btnBox);

    var hint = document.createElement('p');
    hint.textContent = 'Press Escape to close without answering';
    hint.style.cssText = 'color:#555;font-size:0.75em;margin-top:16px;';
    card.appendChild(hint);

    overlay.appendChild(card);
    document.body.appendChild(overlay);

    this._escHandler = function(e) { if (e.key === 'Escape') self._close(overlay); };
    document.addEventListener('keydown', this._escHandler);
  },

  _handleAnswer: function(npcId, question, choiceIndex, overlay) {
    var correct = QuizEngine.check(question, choiceIndex);
    if (correct) {
      ScoreTracker.mark(npcId);
      HUD.update();
      if (ScoreTracker.allDone()) {
        this._close(overlay);
        this._showVictory();
        return;
      }
    }
    var msg = correct
      ? 'Correct! You have mastered this concept!'
      : 'Not quite — review the lesson above and try again!';
    var color = correct ? '#4CAF50' : '#f44336';
    this._close(overlay);
    this._showMessage(npcId, msg, color);
  },

  _showMessage: function(npcId, text, borderColor) {
    var self = this;
    var overlay = document.createElement('div');
    overlay.id = 'quiz-overlay';
    overlay.style.cssText = 'position:fixed;top:0;left:0;width:100%;height:100%;background:rgba(0,0,0,0.7);display:flex;align-items:center;justify-content:center;z-index:10000;font-family:Segoe UI,sans-serif;';

    var box = document.createElement('div');
    box.style.cssText = 'background:#1a1a2e;color:#fff;border-radius:16px;padding:32px;max-width:420px;width:85%;text-align:center;border:2px solid ' + borderColor + ';box-shadow:0 0 30px ' + borderColor + '40;';

    var icon = document.createElement('div');
    icon.style.cssText = 'font-size:3em;margin-bottom:12px;';
    icon.textContent = borderColor === '#4CAF50' ? '\u2705' : '\u274C';
    box.appendChild(icon);

    var p = document.createElement('p');
    p.textContent = text;
    p.style.cssText = 'font-size:1.15em;line-height:1.5;margin:0 0 16px 0;';
    box.appendChild(p);

    var btn = document.createElement('button');
    btn.textContent = 'OK';
    btn.style.cssText = 'padding:10px 32px;border:none;border-radius:8px;background:' + borderColor + ';color:#fff;font-size:1em;cursor:pointer;font-weight:bold;';
    btn.onclick = function() { self._close(overlay); };
    box.appendChild(btn);

    overlay.appendChild(box);
    document.body.appendChild(overlay);
    this._escHandler = function(e) { if (e.key === 'Escape') self._close(overlay); };
    document.addEventListener('keydown', this._escHandler);
  },

  _showVictory: function() {
    var overlay = document.createElement('div');
    overlay.id = 'quiz-overlay';
    overlay.style.cssText = 'position:fixed;top:0;left:0;width:100%;height:100%;background:rgba(0,0,0,0.9);display:flex;align-items:center;justify-content:center;z-index:10000;font-family:Segoe UI,sans-serif;';

    var box = document.createElement('div');
    box.style.cssText = 'background:linear-gradient(135deg,#0d0d2b,#1a1a4e);color:#fff;border-radius:20px;padding:40px;max-width:500px;width:90%;text-align:center;border:3px solid #ffd700;box-shadow:0 0 60px rgba(255,215,0,0.4);';

    var trophy = document.createElement('div');
    trophy.style.cssText = 'font-size:4em;margin-bottom:8px;';
    trophy.textContent = '\uD83C\uDFC6';
    box.appendChild(trophy);

    var title = document.createElement('h1');
    title.textContent = 'MASTER OF ABSTRACTION';
    title.style.cssText = 'color:#ffd700;font-size:2em;margin:0 0 10px 0;';
    box.appendChild(title);

    var sub1 = document.createElement('p');
    sub1.textContent = 'You answered all 5 NPC challenges!';
    sub1.style.cssText = 'font-size:1.2em;color:#ccc;margin-bottom:8px;';
    box.appendChild(sub1);

    var sub2 = document.createElement('p');
    sub2.textContent = 'You truly understand Data Abstraction — lists, indexing, loops, complexity management, and AP concepts.';
    sub2.style.cssText = 'font-size:1em;color:#aaa;margin-bottom:20px;';
    box.appendChild(sub2);

    var btn = document.createElement('button');
    btn.textContent = 'Continue Exploring';
    btn.style.cssText = 'padding:12px 40px;border:none;border-radius:10px;background:#ffd700;color:#000;font-size:1.1em;cursor:pointer;font-weight:bold;';
    btn.onclick = function() { overlay.remove(); };
    box.appendChild(btn);

    overlay.appendChild(box);
    document.body.appendChild(overlay);
  },

  _close: function(overlay) {
    if (overlay && overlay.parentNode) overlay.remove();
    if (this._escHandler) document.removeEventListener('keydown', this._escHandler);
    this.active = false;
  }
};

// ═══════════════════════════════════════════════════════════
//  SRP Module 4: HUD — heads-up display for score
// ═══════════════════════════════════════════════════════════
var HUD = {
  el: null,

  create: function() {
    var old = document.getElementById('abstraction-hud');
    if (old) old.remove();

    this.el = document.createElement('div');
    this.el.id = 'abstraction-hud';
    this.el.style.cssText = 'position:fixed;top:12px;right:12px;z-index:9999;background:linear-gradient(135deg,#1a1a2e,#16213e);border:2px solid #6c63ff;border-radius:12px;padding:10px 18px;color:#fff;font-family:Segoe UI,sans-serif;box-shadow:0 0 20px rgba(100,100,255,0.3);min-width:180px;';
    document.body.appendChild(this.el);
    this.update();
  },

  update: function() {
    if (!this.el) return;
    var s = ScoreTracker.score();
    var filled = '';
    var empty = '';
    for (var i = 0; i < s; i++) filled += '\u2B50';
    for (var j = 0; j < 5 - s; j++) empty += '\u2606';

    var titleDiv = document.createElement('div');
    titleDiv.style.cssText = 'font-weight:bold;font-size:1.1em;margin-bottom:4px;';
    titleDiv.textContent = 'Data Quest';

    var starsDiv = document.createElement('div');
    starsDiv.style.cssText = 'font-size:1.3em;letter-spacing:2px;';
    starsDiv.textContent = filled + empty;

    var countDiv = document.createElement('div');
    countDiv.style.cssText = 'color:#aaa;font-size:0.85em;margin-top:4px;';
    countDiv.textContent = s + '/5 NPCs completed';

    this.el.innerHTML = '';
    this.el.appendChild(titleDiv);
    this.el.appendChild(starsDiv);
    this.el.appendChild(countDiv);
  },

  destroy: function() {
    if (this.el) { this.el.remove(); this.el = null; }
  }
};

// ═══════════════════════════════════════════════════════════
//  SRP Module 5: DataAbstractionLevel — composes everything
// ═══════════════════════════════════════════════════════════
class DataAbstractionLevel {
  constructor(gameEnv) {
    var path = gameEnv.path;
    var width = gameEnv.innerWidth;
    var height = gameEnv.innerHeight;

    ScoreTracker.reset();
    HUD.create();

    var bgData = {
      name: 'data_kingdom',
      greeting: 'Welcome to the Data Kingdom! Speak to each NPC to prove your abstraction knowledge.',
      src: path + '/images/gamify/city.png',
      pixels: { height: 654, width: 966 }
    };

    var SCALE = 5;
    var playerData = {
      id: 'Scholar',
      greeting: 'I seek the wisdom of Data Abstraction!',
      src: path + '/images/gamify/chillguy.png',
      SCALE_FACTOR: SCALE,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: width / 2 - 30, y: height - (height / SCALE) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down:      { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
      downLeft:  { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
      left:      { row: 2, start: 0, columns: 3 },
      right:     { row: 1, start: 0, columns: 3 },
      up:        { row: 3, start: 0, columns: 3 },
      upLeft:    { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
      upRight:   { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    function npcData(id, greeting, sprite, px, initPos, scaleFactor, orient, downAnim) {
      return {
        id: id,
        greeting: greeting,
        src: path + sprite,
        SCALE_FACTOR: scaleFactor,
        ANIMATION_RATE: 100,
        pixels: px,
        INIT_POSITION: initPos,
        orientation: orient,
        down: downAnim,
        hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
        reaction: function() {
          if (ScoreTracker.isComplete(id)) {
            alert(id + ': Already conquered! Move to the next NPC.');
          } else {
            alert(greeting);
          }
        },
        interact: function() {
          QuizOverlay.show(id, greeting);
        }
      };
    }

    var listKeeper = npcData(
      'List Keeper',
      'I am the List Keeper! Lists are ordered sequences of elements. Can you prove you understand them?',
      '/images/gamify/wizard.png',
      { width: 163, height: 185 },
      { x: width * 0.10, y: height * 0.20 },
      3,
      { rows: 1, columns: 1 },
      { row: 0, start: 0, columns: 1 }
    );

    var indexOracle = npcData(
      'Index Oracle',
      'Beep boop! I am the Index Oracle. Do you know how indexing works in Python and JavaScript?',
      '/images/gamify/robot.png',
      { width: 627, height: 316 },
      { x: width * 0.80, y: height * 0.15 },
      8,
      { rows: 3, columns: 6 },
      { row: 1, start: 0, columns: 6 }
    );

    var loopSage = npcData(
      'Loop Sage',
      'Meow-code! I am the Loop Sage. Loops and lists together are powerful — let me test you!',
      '/images/gamify/octocat.png',
      { width: 801, height: 301 },
      { x: width * 0.45, y: height * 0.40 },
      6,
      { rows: 1, columns: 1 },
      { row: 0, start: 0, columns: 1 }
    );

    var abstractArch = npcData(
      'Abstraction Architect',
      'I am the Architect of Abstraction! Treating collections as single units manages complexity. Prove it!',
      '/images/gamify/tux.png',
      { width: 256, height: 352 },
      { x: width * 0.15, y: height * 0.65 },
      4,
      { rows: 1, columns: 1 },
      { row: 0, start: 0, columns: 1 }
    );

    var apGuardian = npcData(
      'AP Guardian',
      'Greetings, future AP scholar! I guard the AP CSP knowledge. Remember: AP lists are 1-indexed!',
      '/images/gamify/nerd.png',
      { width: 343, height: 503 },
      { x: width * 0.78, y: height * 0.60 },
      4,
      { rows: 1, columns: 1 },
      { row: 0, start: 0, columns: 1 }
    );

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: listKeeper },
      { class: Npc, data: indexOracle },
      { class: Npc, data: loopSage },
      { class: Npc, data: abstractArch },
      { class: Npc, data: apGuardian }
    ];
  }
}

export var gameLevelClasses = [DataAbstractionLevel];
export { GameControl };

<IPython.core.display.Javascript object>

## Learning Objectives

> 3.A Generalize data sources through variables.  
> 
> 3.B Use abstraction to manage complexity in a program.
> 
> 3.C Explain how abstraction manages complexity.

## Data Abstractions

Data abstraction means *treating collections of values as a single unit* (like a list, array, or string), so you don’t have to think about all the small details every time you use them. This manages complexity by letting us use a name (like `numbers`) instead of repeatedly referencing each individual element.

For example, in Python:

```py
# A list in Python
numbers = [10, 20, 30, 40]

# Access by index (0-based in Python)
print(numbers[0]) # 10
print(numbers[2]) # 30

# Strings work similarly
word = "hello"
print(word[1]) # 'e'
```

or in JavaScript:

```js
// An array in JavaScript
let numbers = [10, 20, 30, 40];

console.log(numbers[0]); // 10
console.log(numbers[2]); // 30

// Strings are also indexable
let word = "hello";
console.log(word[1]); // 'e'
```

## Using Abstractions to Manage Complexity

Instead of writing code for each item, we use loops and functions to work with lists.

For example, in Python, we could iterate over a list and find the average of its elements:

```py
scores = [85, 92, 78, 90]

# Sum with a loop
total = 0
for score in scores:
    total += score
print("Average:", total / len(scores)) # 86.25
```

and in JS:

```js
let scores = [85, 92, 78, 90];

// Using a loop
let total = 0;
for (let score of scores) {
    total += score;
}
console.log("Average:", total / scores.length);
```

## How Abstraction Manages Complexity

Instead of thinking about *every score individually*, you can use the abstraction "list of scores."
- With a name (`scores`), you manipulate the entire group.
- You don't need to worry about how the list is stored in memory.
- Functions and loops let you work with the *concept* of a collection, not every detail.

What we mean by the last part may be a bit more understandable in some examples of benefits.
- Easier maintenance (add/remove scores without rewriting logic over and over again).
- Reusable functions (average can work for any list of numbers).
- Clearer code (you read "average of scores" instead of `score1 + score2 + score3 + ...`).

## Addressing the Nuances

<style>
  .nu-card { background: linear-gradient(145deg,#3a506b,#1b263b); padding:10px; border:2px solid #5bc0de; border-radius:9px; width:90px; text-align:center; cursor:grab; user-select:none; }
  .nu-drop-zone { min-height:60px; border: 2px dashed #76e2ff; border-radius:10px; padding:10px; background:#112240; display:flex; justify-content:center; align-items:center; }
  .nu-feedback { margin-top: 12px; font-weight:700; }
</style>

<p><strong>TRUE or FALSE: Python and JS use 1 indexing.</strong></p>

<div style="display:flex; gap:20px; align-items:center; margin:20px 0;">
  <div>
    <div class="nu-card" id="true-card" draggable="true">TRUE</div>
  </div>
  <div>
    <div class="nu-card" id="false-card" draggable="true">FALSE</div>
  </div>
  <div>
    <div class="nu-drop-zone" id="answer-box">Drop your answer here</div>
  </div>
</div>

<button id="submit-answer" style="padding:8px 14px;border:none;border-radius:8px;background:#00b8d4;color:#050505;cursor:pointer;">Submit Answer</button>

<div class="nu-feedback" id="nu-feedback"></div>

<script>
(function(){
  let selectedAnswer = null;

  document.getElementById('true-card').addEventListener('dragstart', (e) => e.dataTransfer.setData('text/plain', 'TRUE'));
  document.getElementById('false-card').addEventListener('dragstart', (e) => e.dataTransfer.setData('text/plain', 'FALSE'));

  const box = document.getElementById('answer-box');
  box.addEventListener('dragover', e => e.preventDefault());
  box.addEventListener('drop', e => {
    e.preventDefault();
    const answer = e.dataTransfer.getData('text/plain');
    selectedAnswer = answer;
    box.innerHTML = `<strong>${answer}</strong>`;
    // Remove the cards after drop
    const trueCard = document.getElementById('true-card');
    const falseCard = document.getElementById('false-card');
    if (trueCard) trueCard.remove();
    if (falseCard) falseCard.remove();
  });

  document.getElementById('submit-answer').addEventListener('click', () => {
    const feedback = document.getElementById('nu-feedback');
    if (!selectedAnswer) {
      feedback.style.color = '#ff7f7f';
      feedback.textContent = 'Please drag TRUE or FALSE into the box first.';
      return;
    }
    if (selectedAnswer === 'FALSE') {
      feedback.style.color = '#9bff95';
      feedback.textContent = 'Correct! Python and JS use 0-based indexing, meaning the first element is at index 0. This 0-based indexing is part of data abstraction because it allows us to focus on the logical position in the sequence without worrying about the underlying memory representation, managing complexity by separating the abstract properties from concrete details.';
    } else {
      feedback.style.color = '#ff7f7f';
      feedback.textContent = 'Incorrect. They use 0-based indexing, not 1-based. The first element is at index 0. This 0-based indexing is part of data abstraction because it allows us to focus on the logical position in the sequence without worrying about the underlying memory representation, managing complexity by separating the abstract properties from concrete details.';
    }
  });
})();
</script>

## 🎲 Data Abstraction Arcade: List Lab Builder

Let's look at College Board's interpretation of data abstraction using a simple game.

### Learning targets covered:
- list = ordered sequence
- element + unique index
- index = natural-number access (0-indexed)
- string = ordered series of chars
- data abstraction separates idea (list name) from representation
- complexity management with a value collection that can be handled as one unit
- abstraction via lists are reusable, maintainable, and handle mixed types

<style>
  #list-lab-wrapper { font-family: 'Segoe UI', sans-serif; margin: 12px 0; }
  #list-lab-details > summary { cursor: pointer; padding: 10px 14px; border-radius: 8px; background: #4dacff; color: #fff; font-weight: 700; }
  #list-lab { color: #fff; background:#0d1b2a; border-radius: 12px; border:2px solid #41a7fb; padding: 16px; margin-top: 10px; }
  #list-lab h3 { color:#76e2ff; }
  .list-lab-grid { display:flex; gap:12px; flex-wrap:wrap; }
  .lab-card { background: linear-gradient(145deg,#3a506b,#1b263b); padding:10px; border:2px solid #5bc0de; border-radius:9px; width:90px; text-align:center; cursor:grab; user-select:none; }
  .drop-zone { min-height:90px; border: 2px dashed #76e2ff; border-radius:10px; padding:10px; background:#112240; flex:1; display:flex; flex-direction:column; gap:6px; }
  .drop-slot { background:#243b55; border:1px solid #71c9ce; border-radius:8px; margin:2px 0; min-height:24px; line-height:24px; font-size:0.95em; color:#d3e0ea; }
  .drop-zone.highlight { border-color:#a0e7ff; box-shadow: 0 0 14px rgba(160, 231, 255, .4); }
  .small-label { font-size: 0.88em; color:#b3d7ff; margin-bottom:5px; }
  #feedback { margin-top: 12px; font-weight:700; }
</style>

<details id="list-lab-details" open>
  <summary>Toggle Data Abstraction Arcade</summary>
  <div id="list-lab">
    <h3>List Lab Builder</h3>
    <p>Step 1: Drag 5 element cards into 0-index slots.</p>
    <p>Step 2: Build the string <strong>"DATA"</strong> with char cards in order.</p>
    <p>Step 3: Generate summary tile and drag it into the abstraction crate.</p>

    <div class="list-lab-grid" style="margin-bottom:14px;">
      <div style="flex:1;">
        <div class="small-label">Element Cards (raw values)</div>
        <div id="element-pool"></div>
      </div>
      <div style="flex:2;">
        <div class="small-label">List container (0-index slots)</div>
        <div id="list-container" class="drop-zone"></div>
      </div>
    </div>

    <div class="list-lab-grid" style="margin-bottom:14px;">
      <div style="flex:1;">
        <div class="small-label">Char cards (string sequence)</div>
        <div id="char-pool"></div>
      </div>
      <div style="flex:2;">
        <div class="small-label">Build string (target: DATA)</div>
        <div id="string-builder" class="drop-zone"></div>
      </div>
    </div>

    <div class="list-lab-grid" style="margin-bottom:14px;">
      <div style="flex:1;">
        <div class="small-label">Summary panel</div>
        <div id="summary-panel" class="drop-zone" style="min-height:90px; justify-content:center; align-items:center;">Create summary tile here</div>
      </div>
      <div style="flex:2;">
        <div class="small-label">Abstraction crate</div>
        <div id="crate" class="drop-zone" style="min-height:120px; justify-content:center; align-items:center;">Drop summary tile here</div>
        <button id="crate-summary" style="margin-top:10px;border:none;border-radius:8px;background:#ffd166;color:#1f1f1f;padding:8px 14px;cursor:pointer;font-weight:700;">Generate summary tile</button>
      </div>
    </div>

    <div id="feedback"></div>
    <button id="validate-btn" style="margin-top:10px;padding:8px 14px;border:none;border-radius:8px;background:#00b8d4;color:#050505;cursor:pointer;">Check mastery</button>
  </div>
</details>

<script>
(function(){
  const elements = [ '🐾', '🍎', '7', 'abc', 'True' ];
  const charSet = [ 'D', 'A', 'T', 'A' ];
  const listData = [null, null, null, null, null];
  const stringData = [null, null, null, null];
  let crateHasSummary = false;

  const pool = document.getElementById('element-pool');
  const container = document.getElementById('list-container');
  const charPool = document.getElementById('char-pool');
  const stringBuilder = document.getElementById('string-builder');
  const crate = document.getElementById('crate');
  const summaryPanel = document.getElementById('summary-panel');
  const feedback = document.getElementById('feedback');

  function createCard(value){
    const c = document.createElement('div');
    c.className='lab-card';
    c.textContent = value;
    c.draggable=true;
    c.dataset.value=value;
    c.addEventListener('dragstart', (e)=> e.dataTransfer.setData('text/plain', value));
    return c;
  }

  function buildPool(){
    pool.innerHTML='';
    elements.forEach((v)=> pool.appendChild(createCard(v,'element')));
  }

  function buildChars(){
    charPool.innerHTML='';
    charSet.forEach((v,index)=>{
      const card = createCard(v,'char');
      card.dataset.index=index;
      charPool.appendChild(card);
    });
  }

  function buildContainer(){
    container.innerHTML='';
    for(let i=0;i<5;i++){
      const slot=document.createElement('div');
      slot.className='drop-slot';
      slot.dataset.index=i;
      slot.textContent=`Slot ${i} (0-index)`;
      slot.addEventListener('dragover', e=>{ e.preventDefault(); slot.classList.add('highlight'); });
      slot.addEventListener('dragleave', ()=> slot.classList.remove('highlight'));
      slot.addEventListener('drop', e=>{
        e.preventDefault(); slot.classList.remove('highlight');
        const value = e.dataTransfer.getData('text/plain');
        if(value){
          const cardNode=Array.from(pool.children).find(c=>c.dataset.value===value);
          if(cardNode){ cardNode.remove(); }
          slot.innerHTML = `<strong>${value}</strong>`;
          listData[i]=value;
        }
      });
      container.appendChild(slot);
    }
  }

  function buildStringSlots(){
    stringBuilder.innerHTML='';
    for(let i=1;i<=4;i++){
      const slot=document.createElement('div');
      slot.className='drop-slot';
      slot.dataset.index=i;
      slot.textContent=`#${i}`;
      slot.addEventListener('dragover', e=>{ e.preventDefault(); slot.classList.add('highlight'); });
      slot.addEventListener('dragleave', ()=> slot.classList.remove('highlight'));
      slot.addEventListener('drop', e=>{
        e.preventDefault(); slot.classList.remove('highlight');
        const ch = e.dataTransfer.getData('text/plain');
        if(ch){
          const cardNode=Array.from(charPool.children).find(c=>c.dataset.value===ch);
          if(cardNode){ cardNode.remove(); }
          slot.innerHTML = `<strong>${ch}</strong>`;
          stringData[i-1]=ch;
        }
      });
      stringBuilder.appendChild(slot);
    }
  }

  document.getElementById('crate-summary').addEventListener('click', ()=>{
    const validList = listData.every(x=>x!==null);
    if(!validList){
      feedback.style.color='#ff7f7f';
      return feedback.textContent='Finish the 0-index list first before creating a summary tile.';
    }
    summaryPanel.innerHTML='';
    const summary = createCard('abstract-list');
    summary.style.background='#ffd166';
    summary.style.color='#1f1f1f';
    summary.addEventListener('dragstart', e=> e.dataTransfer.setData('text/plain','abstract-list'));
    summaryPanel.appendChild(summary);
    crateHasSummary = false;
    crate.textContent = 'Drop summary tile here';
    crate.style.borderColor = '#76e2ff';
    feedback.style.color='#9bff95';
    feedback.textContent='Summary tile generated. Drag it from the summary panel into the crate to use abstraction.';
  });

  crate.addEventListener('dragover', e=>{ e.preventDefault(); crate.classList.add('highlight'); });
  crate.addEventListener('dragleave', ()=> crate.classList.remove('highlight'));
  crate.addEventListener('drop', e=>{
    e.preventDefault(); crate.classList.remove('highlight');
    const value = e.dataTransfer.getData('text/plain');
    if(value==='abstract-list') {
      crateHasSummary = true;
      crate.innerHTML = '<strong>ABSTRACTION ACTIVE</strong>';
      crate.style.borderColor='#79c267';
    }
  });

  function resetGame(){
    buildPool();
    buildContainer();
    buildChars();
    buildStringSlots();
    listData.fill(null);
    stringData.fill(null);
    crateHasSummary = false;
    summaryPanel.innerHTML='';
    crate.textContent='Drop summary tile here';
    crate.style.borderColor='#76e2ff';
    feedback.textContent='Game reset. Start with 0-index list building.';
    feedback.style.color='#fff';
  }

  document.getElementById('validate-btn').addEventListener('click', ()=>{
    const listComplete = listData.every(v=>v!==null);
    const stringCorrect = stringData.join('')==='DATA';
    const abstractionDone = crateHasSummary;

    if(!listComplete){
      feedback.style.color='#ff7f7f';
      return feedback.textContent='Missing list elements: enter all 5 elements into 0-index slots.';
    }
    if(!stringCorrect){
      feedback.style.color='#ffd47a';
      return feedback.textContent='String path is not DATA yet; arrange the char cards in order.';
    }
    if(!abstractionDone){
      feedback.style.color='#ffd47a';
      return feedback.textContent='Drag the abstract-list tile into the crate to complete abstraction.';
    }

    feedback.style.color='#9bff95';
    feedback.textContent='Excellent! You demonstrated 0-indexed list ordering, ordered string sequence, and data abstraction through an abstract representation.';
  });

  const resetBtn = document.createElement('button');
  resetBtn.textContent = 'Reset Arcade';
  resetBtn.style.cssText = 'margin-left:10px;padding:8px 14px;border:none;border-radius:8px;background:#ff7f50;color:#1f1f1f;cursor:pointer;font-weight:700;';
  resetBtn.addEventListener('click', resetGame);
  document.getElementById('validate-btn').insertAdjacentElement('afterend', resetBtn);

  resetGame();
})();
</script>